# Imports

In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import *
from src.hmm import HMM
from src.analysis import *
from src.viterbi import viterbi

import math
import random
from scipy.stats import ttest_ind
from pprint import pprint

# Train HMM

In [3]:
states = ["A", "N"]
filename = "../data/GCF_000001405.40_GRCh38.p14_cds_from_genomic.fna.gz"
hmm = HMM(states)
hmm.initialize_parameters()
hmm.train_emission_probs_from_fasta(fasta_filename=filename)

# Synthetic Data

In [10]:
def generate_sample_sequence(codons, probs, n):
    return "".join(random.choices(codons, weights=probs, k=n))
    

codon_list = generate_all_codons()
probs = [math.exp(hmm.emission_probs["A"][codon]) for codon in codon_list]

human_like_seq = {}
random_seq = {}

for i in range(1, 21):
    human_like_seq[f"h{i}"] = generate_sample_sequence(codon_list, probs, 20)
    random_seq[f"r{i}"] = generate_sample_sequence(codon_list, [1/64]*64, 20)


print("Human-like sequences:")
pprint(human_like_seq, sort_dicts=False)
print("Random sequences:")
pprint(random_seq, sort_dicts=False)

Human-like sequences:
{'h1': 'ACAGTGGCGTACGATACTGTTCAAATGAAAAGATTCTCCGAGCAGGAAGACAATCCTAAT',
 'h2': 'CACAAACACGGAGCTGCACCGCTCCCCTTTGGTCAAATGACACTGCTGAACCCTAAAGCG',
 'h3': 'AATAGCTCCCTCCAGGGAAAACAGAAGACTACCGCCAGGCCACCTGCCTTCAGCCCATCT',
 'h4': 'CAGGTCACAAAAGAACCGATTACATTCCCCGTTCACGCCCTAGATTTTCTCAATAACGAA',
 'h5': 'CTTCTGGGATGGCCTTTACCAAGGGAGCTGGGTAAGCAGCATATATTGCACATCGATGGG',
 'h6': 'ACCATGGCACGAGCTAGTTGTGGGCCCGAGCACGACTCTGAGGTTGAGCTGTTTAGTGAA',
 'h7': 'ATGCAGTACAAGCCTCTGCAAGATGATCTGCCACGCGATTCCGGATCAGAGACGACGCAT',
 'h8': 'GCAGCTCGATGCTTACAGCCTCAGCCTCTTAGACTCCCCACAAGCGAGCAGCAGGAATCA',
 'h9': 'GTTAAGGTGGAACGGGGGGCCCTTGTTGCCCAAATGGAAACCGACGGACTGTTAGCTAAG',
 'h10': 'CCAAGCAAAGTCTTTATTCCTTTACAAATCGCAATTGGGTTCGTGGAGGTGTATCAAAAA',
 'h11': 'GAGGGGCAGCTGAGATTTGTAAAGACTGAAACAGTTTTCGGAAAACTCAACATAAGAGGG',
 'h12': 'ACTAGCCACTCGTCCGATGTACTGATGGAGTCCGACGTGTCTCAGGAAGAGGTGAATCCG',
 'h13': 'CAGTTTAACTTCAAGAGGACATCTAGCGACACTTCCAACCCACTGTCAGGCCAACCTGCT',
 'h14': 'CTGGCTTTCACTGGGGGCGAGAGAGGGGGGCATATACGGCGA

# Test with synthetic data

In [11]:
results_human = analyze_genes(human_like_seq, hmm)
results_random = analyze_genes(random_seq, hmm)

human_scores = []
random_scores = []
for score, _ in results_human.values():
    human_scores.append(score)

for score, _ in results_random.values():
    random_scores.append(score)

human_avg = sum(human_scores)/len(human_scores)
random_avg = sum(random_scores)/len(random_scores)

print("Human sequences:")
pprint(results_human, sort_dicts=False)
print("Random sequences:")
pprint(results_random, sort_dicts=False)

print(f"Human-like average: {human_avg}")
print(f"Random average: {random_avg}")

Human sequences:
{'h1': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h2': (0.95, 'AAAAAAAAAAAAAAAAAAAN'),
 'h3': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h4': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h5': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h6': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h7': (0.85, 'AAAAAAAAAAAAAAAAANNN'),
 'h8': (0.75, 'NNNNNAAAAAAAAAAAAAAA'),
 'h9': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h10': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h11': (0.85, 'AAAAAAAAAAAAAAAAANNN'),
 'h12': (0.6, 'NNNNNNNAAAAAAAAAAAAN'),
 'h13': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h14': (0.35, 'AAAAAAANNNNNNNNNNNNN'),
 'h15': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h16': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h17': (0.5, 'NNNNNNNNNNAAAAAAAAAA'),
 'h18': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h19': (1.0, 'AAAAAAAAAAAAAAAAAAAA'),
 'h20': (1.0, 'AAAAAAAAAAAAAAAAAAAA')}
Random sequences:
{'r1': (0.4, 'NNNNNNNNNNNNAAAAAAAA'),
 'r2': (0.0, 'NNNNNNNNNNNNNNNNNNNN'),
 'r3': (0.0, 'NNNNNNNNNNNNNNNNNNNN'),
 'r4': (0.85, 'AAAAAAAAAAAAAAAAANNN'),
 'r5': (0.5, 'NNNNNAAAAAAAAAANNNNN')

# Statistical test

In [12]:
t_stat, p_val = ttest_ind(human_scores, random_scores)

print("t-stat:", t_stat)
print("p-value:", p_val)

t-stat: 6.375789677732162
p-value: 1.7394310761949184e-07
